# Advanced Embedding Generation with Stride and Chunking

## Overview
This notebook generates embeddings for commit diffs using CodeBERT with an advanced stride-based chunking strategy instead of simple mean pooling.

### Features
- **Per-project embeddings**: Separate embedding files for each project
- **Dual representations**: 
  - Full 768-dimensional CodeBERT embeddings
  - Dimensionally-reduced 32-dimensional embeddings
- **Stride-based chunking**: Intelligent chunking with stride parameter to capture multiple overlapping views of diffs
- **Efficient processing**: Batch processing and caching for reproducibility

### Embeddings Generation Strategy
Instead of simple mean pooling:
1. Split each diff into chunks of max 512 tokens
2. Apply stride to create overlapping windows
3. Generate embeddings for each chunk
4. Aggregate chunk embeddings using weighted pooling (stride-aware)
5. Reduce dimensionality with PCA for 32-dim version

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

# Configuration
CONFIG = {
    'projects_dir': '../data/apachejit/projects',
    'diffs_csv': '../data/apachejit/apachejit_with_diffs_v2.csv',
    'output_dir': '../data/apachejit/embeddings',
    'model_name': 'microsoft/codebert-base',
    'chunk_size': 512,  # Max tokens per chunk
    'stride': 256,  # Token stride for overlapping chunks
    'batch_size': 32,
    'full_dim': 768,  # CodeBERT-base dimension
    'reduced_dim': 32,  # Target reduction dimension
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'seed': 42
}

# Set seed for reproducibility
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'full_768d'), exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'reduced_32d'), exist_ok=True)

print("=" * 80)
print("EMBEDDING GENERATION WITH STRIDE AND CHUNKING")
print("=" * 80)
print(f"\n📋 Configuration:")
print(f"   Projects directory: {CONFIG['projects_dir']}")
print(f"   Diffs CSV: {CONFIG['diffs_csv']}")
print(f"   Output directory: {CONFIG['output_dir']}")
print(f"   Model: {CONFIG['model_name']}")
print(f"   Chunk size: {CONFIG['chunk_size']} tokens")
print(f"   Stride: {CONFIG['stride']} tokens")
print(f"   Full dimension: {CONFIG['full_dim']}")
print(f"   Reduced dimension: {CONFIG['reduced_dim']}")
print(f"   Device: {CONFIG['device']}\n")

## Section 1: Load CodeBERT Model and Tokenizer

In [ ]:
print("Loading CodeBERT model...")
try:
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    model = AutoModel.from_pretrained(CONFIG['model_name'])
    model.to(CONFIG['device'])
    model.eval()
    print(f"✓ CodeBERT loaded from {CONFIG['model_name']}")
    print(f"  Model output dimension: {model.config.hidden_size}")
except Exception as e:
    print(f"✗ Error loading CodeBERT: {e}")
    raise

## Section 2: Load and Prepare Data

In [ ]:
print("Loading diffs and project data...")

# Load diffs CSV
print(f"  Loading {CONFIG['diffs_csv']}...")
df_diffs = pd.read_csv(CONFIG['diffs_csv'])
print(f"  ✓ Loaded {len(df_diffs)} rows")

# Get unique projects
projects = sorted(df_diffs['project'].unique())
print(f"  ✓ Found {len(projects)} unique projects:")
for proj in projects[:10]:
    n_commits = len(df_diffs[df_diffs['project'] == proj])
    print(f"     • {proj}: {n_commits} commits")
if len(projects) > 10:
    print(f"     ... and {len(projects) - 10} more projects")

# Verify project files exist
print(f"\n  Verifying project files in {CONFIG['projects_dir']}...")
existing_projects = []
for proj in projects:
    proj_file = os.path.join(CONFIG['projects_dir'], f"{proj}.csv")
    if os.path.exists(proj_file):
        existing_projects.append(proj)
        
print(f"  ✓ {len(existing_projects)} project CSV files found")
if len(existing_projects) < len(projects):
    print(f"  ⚠ Warning: {len(projects) - len(existing_projects)} projects missing CSV files")

print("\n✓ Data loaded successfully\n")

## Section 3: Implement Stride-Based Chunking Strategy

In [ ]:
def create_chunks_with_stride(tokens, chunk_size=512, stride=256):
    """
    Create overlapping chunks of tokens using stride.
    
    Args:
        tokens: List of token IDs
        chunk_size: Maximum tokens per chunk
        stride: Number of tokens to move for next chunk (overlap = chunk_size - stride)
        
    Returns:
        List of token chunks and their metadata (start_idx, end_idx)
    """
    chunks = []
    if len(tokens) <= chunk_size:
        # If text is shorter than chunk size, return single chunk
        chunks.append({
            'tokens': tokens,
            'start_idx': 0,
            'end_idx': len(tokens),
            'is_first': True,
            'is_last': True
        })
    else:
        # Create overlapping chunks
        i = 0
        chunk_num = 0
        while i < len(tokens):
            end_idx = min(i + chunk_size, len(tokens))
            is_last = (end_idx == len(tokens))
            
            chunks.append({
                'tokens': tokens[i:end_idx],
                'start_idx': i,
                'end_idx': end_idx,
                'is_first': (chunk_num == 0),
                'is_last': is_last,
                'chunk_num': chunk_num
            })
            
            if is_last:
                break
                
            i += stride
            chunk_num += 1
    
    return chunks

def aggregate_chunk_embeddings(chunk_embeddings, chunk_metadata, strategy='stride_weighted'):
    """
    Aggregate embeddings from multiple chunks into single embedding.
    
    Args:
        chunk_embeddings: Array of shape (n_chunks, embedding_dim)
        chunk_metadata: List of metadata dicts for each chunk
        strategy: Aggregation strategy - 'mean', 'first', 'last', or 'stride_weighted'
        
    Returns:
        Aggregated embedding of shape (embedding_dim,)
    """
    if len(chunk_embeddings) == 1:
        # Single chunk, return as is
        return chunk_embeddings[0]
    
    if strategy == 'mean':
        return np.mean(chunk_embeddings, axis=0)
    
    elif strategy == 'first':
        return chunk_embeddings[0]
    
    elif strategy == 'last':
        return chunk_embeddings[-1]
    
    elif strategy == 'stride_weighted':
        # Weight chunks based on their position and overlap
        # Earlier chunks get slightly higher weight, especially first chunk
        weights = np.ones(len(chunk_embeddings))
        weights[0] = 1.5  # Emphasize first chunk (beginning of diff)
        weights[-1] = 1.2  # Slightly emphasize last chunk (end of diff)
        
        # Normalize weights
        weights = weights / weights.sum()
        
        return np.average(chunk_embeddings, axis=0, weights=weights)
    
    else:
        return np.mean(chunk_embeddings, axis=0)

def generate_embedding_with_stride(diff_text, tokenizer, model, device, 
                                   chunk_size=512, stride=256, aggregation='stride_weighted'):
    """
    Generate embedding for a diff using stride-based chunking.
    
    Args:
        diff_text: Text of the diff
        tokenizer: CodeBERT tokenizer
        model: CodeBERT model
        device: Device to use (cuda or cpu)
        chunk_size: Max tokens per chunk
        stride: Stride for overlapping chunks
        aggregation: Aggregation strategy for chunks
        
    Returns:
        Embedding of shape (768,) - CodeBERT output dimension
    """
    if not isinstance(diff_text, str) or len(diff_text.strip()) == 0:
        # Return zero embedding for empty text
        return np.zeros(768, dtype=np.float32)
    
    try:
        # Tokenize the entire diff
        encodings = tokenizer(diff_text, truncation=False, return_tensors='pt', 
                             padding=False, add_special_tokens=True)
        tokens = encodings['input_ids'][0].tolist()
        
        # Create chunks with stride
        chunks = create_chunks_with_stride(tokens, chunk_size=chunk_size, stride=stride)
        
        # Generate embeddings for each chunk
        chunk_embeddings = []
        chunk_metas = []
        
        with torch.no_grad():
            for chunk_info in chunks:
                chunk_tokens = chunk_info['tokens']
                
                # Convert tokens to tensor
                input_ids = torch.tensor([chunk_tokens], device=device)
                
                # Get embeddings
                outputs = model(input_ids)
                last_hidden = outputs.last_hidden_state  # Shape: (1, seq_len, 768)
                
                # Mean pooling over sequence dimension (excluding padding)
                sequence_output = last_hidden[0]  # Shape: (seq_len, 768)
                pooled = sequence_output.mean(dim=0)  # Shape: (768,)
                
                chunk_embeddings.append(pooled.cpu().numpy())
                chunk_metas.append(chunk_info)
        
        # Aggregate chunk embeddings
        chunk_embeddings = np.array(chunk_embeddings)  # Shape: (n_chunks, 768)
        aggregated = aggregate_chunk_embeddings(chunk_embeddings, chunk_metas, strategy=aggregation)
        
        return aggregated.astype(np.float32)
        
    except Exception as e:
        print(f"  ✗ Error generating embedding: {e}")
        return np.zeros(768, dtype=np.float32)

print("✓ Stride-based chunking functions implemented")

## Section 4: Load Data and Project Information

In [ ]:
# Load the main dataset with diffs
print("Loading apachejit_with_diffs_v2.csv...")
diffs_df = pd.read_csv(CONFIG['diffs_csv'])
print(f"✓ Loaded {len(diffs_df)} rows")
print(f"✓ Columns: {diffs_df.columns.tolist()}")

# Get list of unique projects
projects = sorted(diffs_df['project_name'].unique())
print(f"\n✓ Found {len(projects)} unique projects:")
for proj in projects:
    count = len(diffs_df[diffs_df['project_name'] == proj])
    print(f"  - {proj}: {count} commits")

# Verify that project CSV files exist
print(f"\nVerifying project CSV files in {CONFIG['projects_dir']}...")
missing_projects = []
for proj in projects:
    proj_path = os.path.join(CONFIG['projects_dir'], f"{proj}.csv")
    if not os.path.exists(proj_path):
        missing_projects.append(proj)
        print(f"  ✗ Missing: {proj_path}")
    else:
        rows = len(pd.read_csv(proj_path))
        print(f"  ✓ {proj}: {rows} rows")

if missing_projects:
    print(f"\n⚠ Warning: {len(missing_projects)} project files missing")
else:
    print(f"\n✓ All project files verified")

# Create commit ID to diff text mapping
print("\nBuilding commit-to-diff mapping...")
commit_diff_map = {}  # commit_id -> diff_text
project_commits = {}  # project_name -> list of commit_ids

for _, row in diffs_df.iterrows():
    commit_id = row['commit_id']
    project = row['project_name']
    diff_text = row['diff_text'] if 'diff_text' in diffs_df.columns else ""
    
    commit_diff_map[commit_id] = diff_text
    
    if project not in project_commits:
        project_commits[project] = []
    project_commits[project].append(commit_id)

# Sort commits within each project to maintain order
for proj in project_commits:
    project_commits[proj] = sorted(set(project_commits[proj]))

print(f"✓ Built mapping for {len(commit_diff_map)} unique commits")
print(f"✓ Projects with commit lists: {len(project_commits)}")
for proj, commits in project_commits.items():
    print(f"  - {proj}: {len(commits)} commits")

## Section 5: Generate Embeddings with Stride-Based Chunking

In [ ]:
# Generate embeddings for all commits across all projects
print("=" * 80)
print("GENERATING EMBEDDINGS WITH STRIDE-BASED CHUNKING")
print("=" * 80)

# Store all embeddings for later PCA fitting
all_embeddings_768 = []
all_commit_ids = []
embeddings_by_project = {}  # project -> {'commit_ids': [...], 'embeddings_768': [...]}

total_commits = sum(len(commits) for commits in project_commits.values())
processed = 0
failed = 0

for project_idx, project_name in enumerate(projects, 1):
    print(f"\n[{project_idx}/{len(projects)}] Processing: {project_name}")
    print("-" * 80)
    
    if project_name not in project_commits:
        print(f"  ✗ No commits found for project")
        continue
    
    commit_ids = project_commits[project_name]
    project_embeddings_768 = []
    valid_commit_ids = []
    
    # Process commits for this project
    for commit_idx, commit_id in enumerate(tqdm(commit_ids, desc=f"  Generating embeddings")):
        if commit_id not in commit_diff_map:
            failed += 1
            continue
        
        diff_text = commit_diff_map[commit_id]
        
        # Generate embedding with stride-based chunking
        embedding = generate_embedding_with_stride(
            diff_text=diff_text,
            tokenizer=tokenizer,
            model=model,
            device=CONFIG['device'],
            chunk_size=CONFIG['chunk_size'],
            stride=CONFIG['stride'],
            aggregation='stride_weighted'
        )
        
        # Skip if embedding is all zeros (failure case)
        if np.any(embedding):
            project_embeddings_768.append(embedding)
            valid_commit_ids.append(commit_id)
            all_embeddings_768.append(embedding)
            all_commit_ids.append(commit_id)
        else:
            failed += 1
        
        processed += 1
    
    # Store project embeddings
    if len(project_embeddings_768) > 0:
        embeddings_by_project[project_name] = {
            'commit_ids': valid_commit_ids,
            'embeddings_768': np.array(project_embeddings_768, dtype=np.float32)
        }
        print(f"  ✓ Generated {len(project_embeddings_768)} embeddings ({len(valid_commit_ids)} commits)")
    else:
        print(f"  ✗ No valid embeddings generated for {project_name}")

print("\n" + "=" * 80)
print(f"SUMMARY: Processed {processed} commits, Failed: {failed}")
print(f"Total valid embeddings collected: {len(all_embeddings_768)}")
print("=" * 80)

## Section 6: Apply PCA for Dimensionality Reduction (768 → 32 dimensions)

In [ ]:
print("=" * 80)
print("APPLYING PCA FOR DIMENSIONALITY REDUCTION")
print("=" * 80)

# Prepare data for PCA
embeddings_768_array = np.array(all_embeddings_768, dtype=np.float32)
print(f"\nOriginal embeddings shape: {embeddings_768_array.shape}")

# Standardize embeddings before PCA
scaler = StandardScaler()
embeddings_768_scaled = scaler.fit_transform(embeddings_768_array)
print(f"Scaled embeddings shape: {embeddings_768_scaled.shape}")
print(f"✓ Scaling: mean={embeddings_768_scaled.mean():.6f}, std={embeddings_768_scaled.std():.6f}")

# Fit PCA
print(f"\nFitting PCA to reduce {CONFIG['full_dim']} → {CONFIG['reduced_dim']} dimensions...")
pca = PCA(n_components=CONFIG['reduced_dim'], random_state=CONFIG['seed'])
embeddings_32_array = pca.fit_transform(embeddings_768_scaled)

print(f"✓ PCA fitted successfully")
print(f"✓ Reduced embeddings shape: {embeddings_32_array.shape}")
print(f"✓ Explained variance ratio sum: {pca.explained_variance_ratio_.sum():.4f}")
print(f"✓ Top 5 explained variance ratios:")
for i, var in enumerate(pca.explained_variance_ratio_[:5], 1):
    print(f"  - PC{i}: {var:.4f}")

# Create mapping from commit_id to reduced embedding index
commit_to_embedding_idx = {commit_id: idx for idx, commit_id in enumerate(all_commit_ids)}

# Create reduced embeddings for each project
print("\nAllocating reduced embeddings to projects...")
for project_name in embeddings_by_project:
    project_commits_list = embeddings_by_project[project_name]['commit_ids']
    
    # Get indices for this project's commits
    indices = [commit_to_embedding_idx[cid] for cid in project_commits_list]
    
    # Get reduced embeddings for these indices
    reduced_embs = embeddings_32_array[indices]
    
    embeddings_by_project[project_name]['embeddings_32'] = reduced_embs.astype(np.float32)
    
    print(f"  ✓ {project_name}: {len(reduced_embs)} reduced embeddings")

print("\n" + "=" * 80)
print(f"✓ PCA reduction complete: {len(embeddings_by_project)} projects with dual embeddings")
print("=" * 80)

## Section 7: Save Per-Project Embedding Files

In [ ]:
print("=" * 80)
print("SAVING PER-PROJECT EMBEDDING FILES")
print("=" * 80)

# Create output directories
output_768_dir = os.path.join(CONFIG['output_dir'], 'full_768d')
output_32_dir = os.path.join(CONFIG['output_dir'], 'reduced_32d')

os.makedirs(output_768_dir, exist_ok=True)
os.makedirs(output_32_dir, exist_ok=True)

print(f"\nOutput directories:")
print(f"  - Full 768d: {output_768_dir}")
print(f"  - Reduced 32d: {output_32_dir}")

# Save embeddings for each project
saved_projects = []
for project_name in sorted(embeddings_by_project.keys()):
    project_data = embeddings_by_project[project_name]
    commit_ids = project_data['commit_ids']
    embeddings_768 = project_data['embeddings_768']
    embeddings_32 = project_data['embeddings_32']
    
    # Save 768-dimensional embeddings
    file_768 = os.path.join(output_768_dir, f"{project_name}.npz")
    np.savez_compressed(
        file_768,
        embeddings=embeddings_768,
        commit_ids=np.array(commit_ids),
        n_commits=len(commit_ids),
        embedding_dim=CONFIG['full_dim'],
        stride_config={
            'chunk_size': CONFIG['chunk_size'],
            'stride': CONFIG['stride'],
            'aggregation': 'stride_weighted'
        }
    )
    
    # Save 32-dimensional embeddings
    file_32 = os.path.join(output_32_dir, f"{project_name}.npz")
    np.savez_compressed(
        file_32,
        embeddings=embeddings_32,
        commit_ids=np.array(commit_ids),
        n_commits=len(commit_ids),
        embedding_dim=CONFIG['reduced_dim'],
        stride_config={
            'chunk_size': CONFIG['chunk_size'],
            'stride': CONFIG['stride'],
            'aggregation': 'stride_weighted'
        },
        pca_info={
            'explained_variance_ratio_sum': str(pca.explained_variance_ratio_.sum()),
            'n_components': CONFIG['reduced_dim'],
            'original_dim': CONFIG['full_dim']
        }
    )
    
    saved_projects.append({
        'project': project_name,
        'n_commits': len(commit_ids),
        'file_768': file_768,
        'file_32': file_32,
        'size_768_mb': os.path.getsize(file_768) / (1024 * 1024),
        'size_32_mb': os.path.getsize(file_32) / (1024 * 1024)
    })
    
    print(f"✓ {project_name}")
    print(f"  - Commits: {len(commit_ids)}")
    print(f"  - 768d file: {os.path.basename(file_768)} ({saved_projects[-1]['size_768_mb']:.2f} MB)")
    print(f"  - 32d file:  {os.path.basename(file_32)} ({saved_projects[-1]['size_32_mb']:.2f} MB)")

# Create summary dataframe
summary_df = pd.DataFrame(saved_projects)

print("\n" + "=" * 80)
print("SAVE SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print(f"\nTotal projects: {len(summary_df)}")
print(f"Total commits: {summary_df['n_commits'].sum()}")
print(f"Total 768d storage: {summary_df['size_768_mb'].sum():.2f} MB")
print(f"Total 32d storage: {summary_df['size_32_mb'].sum():.2f} MB")
print(f"Compression ratio: {(summary_df['size_768_mb'].sum() / summary_df['size_32_mb'].sum()):.2f}x")
print("=" * 80)

## Section 8: Verification and Summary Statistics

In [ ]:
print("=" * 80)
print("VERIFICATION: LOADING AND INSPECTING SAVED EMBEDDINGS")
print("=" * 80)

verification_results = []

for project_name in sorted(embeddings_by_project.keys()):
    print(f"\n[{project_name}]")
    
    # Load 768d embeddings
    file_768 = os.path.join(output_768_dir, f"{project_name}.npz")
    data_768 = np.load(file_768, allow_pickle=True)
    
    emb_768 = data_768['embeddings']
    commit_ids_768 = data_768['commit_ids']
    
    # Load 32d embeddings
    file_32 = os.path.join(output_32_dir, f"{project_name}.npz")
    data_32 = np.load(file_32, allow_pickle=True)
    
    emb_32 = data_32['embeddings']
    commit_ids_32 = data_32['commit_ids']
    
    # Verify consistency
    assert len(emb_768) == len(emb_32), f"Dimension mismatch for {project_name}"
    assert len(commit_ids_768) == len(commit_ids_32), f"Commit ID count mismatch for {project_name}"
    assert (commit_ids_768 == commit_ids_32).all(), f"Commit ID order mismatch for {project_name}"
    
    print(f"  ✓ 768d embeddings: {emb_768.shape}")
    print(f"  ✓ 32d embeddings: {emb_32.shape}")
    print(f"  ✓ Commits: {len(commit_ids_768)}")
    print(f"  ✓ 768d range: [{emb_768.min():.4f}, {emb_768.max():.4f}]")
    print(f"  ✓ 32d range: [{emb_32.min():.4f}, {emb_32.max():.4f}]")
    
    # Compute embedding statistics
    norm_768 = np.linalg.norm(emb_768, axis=1)
    norm_32 = np.linalg.norm(emb_32, axis=1)
    
    verification_results.append({
        'project': project_name,
        'n_commits': len(commit_ids_768),
        'dim_768': emb_768.shape[1],
        'dim_32': emb_32.shape[1],
        'norm_768_mean': norm_768.mean(),
        'norm_768_std': norm_768.std(),
        'norm_32_mean': norm_32.mean(),
        'norm_32_std': norm_32.std()
    })

# Print comprehensive summary
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

verification_df = pd.DataFrame(verification_results)
print(verification_df.to_string(index=False))

print(f"\nGlobal Statistics:")
print(f"  - Total projects: {len(verification_df)}")
print(f"  - Total commits: {verification_df['n_commits'].sum()}")
print(f"  - Avg commits/project: {verification_df['n_commits'].mean():.1f}")
print(f"  - Min commits/project: {verification_df['n_commits'].min()}")
print(f"  - Max commits/project: {verification_df['n_commits'].max()}")

print(f"\nEmbedding Statistics (768d):")
print(f"  - Mean norm: {verification_df['norm_768_mean'].mean():.4f} ± {verification_df['norm_768_std'].mean():.4f}")
print(f"  - Min norm: {verification_df['norm_768_mean'].min():.4f}")
print(f"  - Max norm: {verification_df['norm_768_mean'].max():.4f}")

print(f"\nEmbedding Statistics (32d):")
print(f"  - Mean norm: {verification_df['norm_32_mean'].mean():.4f} ± {verification_df['norm_32_std'].mean():.4f}")
print(f"  - Min norm: {verification_df['norm_32_mean'].min():.4f}")
print(f"  - Max norm: {verification_df['norm_32_mean'].max():.4f}")

print(f"\nPCA Statistics:")
print(f"  - Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")
print(f"  - Reduction ratio: {CONFIG['full_dim']} → {CONFIG['reduced_dim']} ({CONFIG['reduced_dim']/CONFIG['full_dim']*100:.1f}%)")

print(f"\nStride-Based Chunking Config:")
print(f"  - Chunk size: {CONFIG['chunk_size']} tokens")
print(f"  - Stride: {CONFIG['stride']} tokens")
print(f"  - Overlap: {CONFIG['chunk_size'] - CONFIG['stride']} tokens ({(1 - CONFIG['stride']/CONFIG['chunk_size'])*100:.1f}%)")
print(f"  - Aggregation: stride_weighted")

print("\n" + "=" * 80)
print("✓ EMBEDDING GENERATION COMPLETE")
print("=" * 80)